In [ ]:
from google.colab import drive

In [ ]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install duckdb

In [ ]:
import duckdb

duckdb.sql("""
    SELECT Period, "Org Code", "Org name", "A&E attendances Type 1"
    FROM read_csv_auto('/content/drive/MyDrive/ae_project/*.csv')
    LIMIT 5
""").show()

┌─────────────────────┬──────────┬─────────────────────────────────────────────────────────┬────────────────────────┐
│       Period        │ Org Code │                        Org name                         │ A&E attendances Type 1 │
│       varchar       │ varchar  │                         varchar                         │         int64          │
├─────────────────────┼──────────┼─────────────────────────────────────────────────────────┼────────────────────────┤
│ MSitAE-OCTOBER-2025 │ RAN      │ ROYAL NATIONAL ORTHOPAEDIC HOSPITAL NHS TRUST           │                      0 │
│ MSitAE-OCTOBER-2025 │ RBQ      │ LIVERPOOL HEART AND CHEST HOSPITAL NHS FOUNDATION TRUST │                      0 │
│ MSitAE-OCTOBER-2025 │ RLQ      │ WYE VALLEY NHS TRUST                                    │                   6138 │
│ MSitAE-OCTOBER-2025 │ RYY      │ KENT COMMUNITY HEALTH NHS FOUNDATION TRUST              │                      0 │
│ MSitAE-OCTOBER-2025 │ AAH      │ TETBURY HOSPITAL TRUS

In [ ]:
duckdb.sql("""
    SELECT Period, COUNT(*) AS rows_in_file
    FROM read_csv_auto('/content/drive/MyDrive/ae_project/*.csv', union_by_name=true)
    GROUP BY Period
    ORDER BY Period
""").show()

┌───────────────────────┬──────────────┐
│        Period         │ rows_in_file │
│        varchar        │    int64     │
├───────────────────────┼──────────────┤
│ MSitAE-APRIL-2026     │          190 │
│ MSitAE-AUGUST-2026    │          191 │
│ MSitAE-DECEMBER-2025  │          197 │
│ MSitAE-FEBRUARY-2026  │          198 │
│ MSitAE-JANUARY-2026   │          197 │
│ MSitAE-JULY-2026      │          191 │
│ MSitAE-JUNE-2026      │          192 │
│ MSitAE-MARCH-2026     │          199 │
│ MSitAE-MAY-2026       │          192 │
│ MSitAE-NOVEMBER-2025  │          197 │
│ MSitAE-OCTOBER-2025   │          197 │
│ MSitAE-SEPTEMBER-2026 │          191 │
│ TOTAL                 │            8 │
│ TOTAL                 │            1 │
│ Total                 │            2 │
├───────────────────────┴──────────────┤
│ 15 rows                    2 columns │
└──────────────────────────────────────┘



In [ ]:
duckdb.sql("""
CREATE OR REPLACE TABLE ae AS
SELECT
    CAST(strptime(
        concat(upper(left(split_part(Period,'-',2),1)),
               lower(substr(split_part(Period,'-',2),2)),
               ' ', split_part(Period,'-',3)),
        '%B %Y') AS DATE)                      AS month,
    "Org Code"                                 AS org_code,
    trim("Org name")                           AS org_name,
    trim("Parent Org")                         AS region,
    "A&E attendances Type 1"                   AS type1_attendances,
    "A&E attendances Type 1" + "A&E attendances Type 2"
      + "A&E attendances Other A&E Department"
      + "A&E attendances Booked Appointments Type 1"
      + "A&E attendances Booked Appointments Type 2"
      + "A&E attendances Booked Appointments Other Department"
                                               AS total_attendances,
    "Attendances over 4hrs Type 1" + "Attendances over 4hrs Type 2"
      + "Attendances over 4hrs Other Department"
      + "Attendances over 4hrs Booked Appointments Type 1"
      + "Attendances over 4hrs Booked Appointments Type 2"
      + "Attendances over 4hrs Booked Appointments Other Department"
                                               AS total_over_4hrs
FROM read_csv_auto('/content/drive/MyDrive/ae_project/*.csv', union_by_name=true)
WHERE Period LIKE 'MSitAE-%'
""")

In [ ]:
duckdb.sql("""
    SELECT month, COUNT(*) AS orgs, SUM(total_attendances) AS attendances
    FROM ae
    GROUP BY month
    ORDER BY month
""").show()

┌────────────┬───────┬─────────────┐
│   month    │ orgs  │ attendances │
│    date    │ int64 │   int128    │
├────────────┼───────┼─────────────┤
│ 2025-10-01 │   197 │     2401266 │
│ 2025-11-01 │   197 │     2347046 │
│ 2025-12-01 │   197 │     2326139 │
│ 2026-01-01 │   197 │     2319296 │
│ 2026-02-01 │   198 │     4234900 │
│ 2026-03-01 │   199 │     2432272 │
│ 2026-04-01 │   190 │     2345329 │
│ 2026-05-01 │   192 │     2457398 │
│ 2026-06-01 │   192 │     2437906 │
│ 2026-07-01 │   191 │     2487580 │
│ 2026-08-01 │   191 │     2342959 │
│ 2026-09-01 │   191 │     2444743 │
├────────────┴───────┴─────────────┤
│ 12 rows                3 columns │
└──────────────────────────────────┘



In [ ]:
duckdb.sql("""
    DELETE FROM ae
    WHERE upper(org_code) = 'TOTAL' OR upper(org_name) = 'TOTAL'
""")

In [ ]:
duckdb.sql("""
    SELECT month, COUNT(*) AS orgs, SUM(total_attendances) AS attendances
    FROM ae
    GROUP BY month
    ORDER BY month
""").show()

┌────────────┬───────┬─────────────┐
│   month    │ orgs  │ attendances │
│    date    │ int64 │   int128    │
├────────────┼───────┼─────────────┤
│ 2025-10-01 │   197 │     2401266 │
│ 2025-11-01 │   197 │     2347046 │
│ 2025-12-01 │   197 │     2326139 │
│ 2026-01-01 │   197 │     2319296 │
│ 2026-02-01 │   197 │     2117450 │
│ 2026-03-01 │   199 │     2432272 │
│ 2026-04-01 │   190 │     2345329 │
│ 2026-05-01 │   192 │     2457398 │
│ 2026-06-01 │   192 │     2437906 │
│ 2026-07-01 │   191 │     2487580 │
│ 2026-08-01 │   191 │     2342959 │
│ 2026-09-01 │   191 │     2444743 │
├────────────┴───────┴─────────────┤
│ 12 rows                3 columns │
└──────────────────────────────────┘



In [ ]:
# Query 1: England as a whole, month by month. This is a sanity check and your first trend. Run it in a new cell:

duckdb.sql("""
    SELECT
        month,
        SUM(total_attendances) AS attendances,
        SUM(total_over_4hrs)   AS over_4hrs,
        ROUND(100.0 * (1 - SUM(total_over_4hrs) / SUM(total_attendances)), 1) AS pct_within_4hrs
    FROM ae
    GROUP BY month
    ORDER BY month
""").show()

┌────────────┬─────────────┬───────────┬─────────────────┐
│   month    │ attendances │ over_4hrs │ pct_within_4hrs │
│    date    │   int128    │  int128   │     double      │
├────────────┼─────────────┼───────────┼─────────────────┤
│ 2025-10-01 │     2401266 │    619874 │            74.2 │
│ 2025-11-01 │     2347046 │    605140 │            74.2 │
│ 2025-12-01 │     2326139 │    608475 │            73.8 │
│ 2026-01-01 │     2319296 │    638938 │            72.5 │
│ 2026-02-01 │     2117450 │    549366 │            74.1 │
│ 2026-03-01 │     2432272 │    556711 │            77.1 │
│ 2026-04-01 │     2345329 │    541542 │            76.9 │
│ 2026-05-01 │     2457398 │    597323 │            75.7 │
│ 2026-06-01 │     2437906 │    608917 │            75.0 │
│ 2026-07-01 │     2487580 │    612579 │            75.4 │
│ 2026-08-01 │     2342959 │    584767 │            75.0 │
│ 2026-09-01 │     2444743 │    626751 │            74.4 │
├────────────┴─────────────┴───────────┴────────────────

In [ ]:
# Query 2: the 10 worst-performing hospital trusts in September 2026. Run this in another cell:

duckdb.sql("""
    SELECT
        org_name,
        region,
        total_attendances,
        ROUND(100.0 * (1 - total_over_4hrs * 1.0 / total_attendances), 1) AS pct_within_4hrs
    FROM ae
    WHERE month = '2026-09-01'
      AND type1_attendances > 0
      AND total_attendances > 0
    ORDER BY pct_within_4hrs ASC
    LIMIT 10
""").show()

┌──────────────────────────────────────────────────────────────┬──────────────────────────────────────┬───────────────────┬─────────────────┐
│                           org_name                           │                region                │ total_attendances │ pct_within_4hrs │
│                           varchar                            │               varchar                │       int64       │     double      │
├──────────────────────────────────────────────────────────────┼──────────────────────────────────────┼───────────────────┼─────────────────┤
│ EAST CHESHIRE NHS TRUST                                      │ NHS ENGLAND NORTH WEST               │              4811 │            49.8 │
│ THE SHREWSBURY AND TELFORD HOSPITAL NHS TRUST                │ NHS ENGLAND MIDLANDS                 │             14335 │            51.0 │
│ SURREY AND SUSSEX HEALTHCARE NHS TRUST                       │ NHS ENGLAND SOUTH EAST               │             11041 │            52.1 │
│ WIRR

In [ ]:
# Cell 1: build the table.

duckdb.sql("""
CREATE OR REPLACE TABLE trust_monthly AS
WITH base AS (
    SELECT month, org_code, org_name, region, total_attendances,
           ROUND(100.0 * (1 - total_over_4hrs * 1.0 / total_attendances), 1) AS pct_within_4hrs
    FROM ae
    WHERE type1_attendances > 0
      AND total_attendances > 0
)
SELECT *,
    LAG(pct_within_4hrs) OVER (PARTITION BY org_code ORDER BY month) AS prev_month_pct,
    ROUND(pct_within_4hrs
          - LAG(pct_within_4hrs) OVER (PARTITION BY org_code ORDER BY month), 1) AS change_vs_prev,
    CASE WHEN pct_within_4hrs < 60 THEN 'Below 60%'
         WHEN pct_within_4hrs < 75 THEN '60-75%'
         ELSE '75% or above' END AS performance_band
FROM base
""")

In [ ]:
# Cell 2: see the biggest drops in September.

duckdb.sql("""
    SELECT org_name, pct_within_4hrs, prev_month_pct, change_vs_prev
    FROM trust_monthly
    WHERE month = '2026-09-01'
    ORDER BY change_vs_prev ASC
    LIMIT 10
""").show()

┌───────────────────────────────────────────────────────────┬─────────────────┬────────────────┬────────────────┐
│                         org_name                          │ pct_within_4hrs │ prev_month_pct │ change_vs_prev │
│                          varchar                          │     double      │     double     │     double     │
├───────────────────────────────────────────────────────────┼─────────────────┼────────────────┼────────────────┤
│ BIRMINGHAM WOMEN'S AND CHILDREN'S NHS FOUNDATION TRUST    │            78.2 │           93.7 │          -15.5 │
│ BARNSLEY HOSPITAL NHS FOUNDATION TRUST                    │            71.3 │           78.9 │           -7.6 │
│ HULL UNIVERSITY TEACHING HOSPITALS NHS TRUST              │            54.5 │           61.4 │           -6.9 │
│ HOMERTON HEALTHCARE NHS FOUNDATION TRUST                  │            78.5 │           83.4 │           -4.9 │
│ EAST CHESHIRE NHS TRUST                                   │            49.8 │         

In [ ]:
# Clean data

import os
os.makedirs('/content/drive/MyDrive/ae_outputs', exist_ok=True)

duckdb.sql("""
    COPY trust_monthly
    TO '/content/drive/MyDrive/ae_outputs/trust_monthly.csv'
    (HEADER, DELIMITER ',')
""")

duckdb.sql("""
    COPY (
        SELECT month,
               SUM(total_attendances) AS attendances,
               ROUND(100.0 * (1 - SUM(total_over_4hrs) / SUM(total_attendances)), 1) AS pct_within_4hrs
        FROM ae GROUP BY month ORDER BY month
    ) TO '/content/drive/MyDrive/ae_outputs/england_monthly.csv'
    (HEADER, DELIMITER ',')
""")

In [ ]:
duckdb.sql("SELECT COUNT(*) FROM trust_monthly").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│         1449 │
└──────────────┘

